In [2]:
import io
import csv
import pandas as pd
from google.colab import files

print("=== Upload your CSV file (e.g., Student_Grade.csv) below ===")
uploaded = files.upload()

if uploaded:
    file_name = list(uploaded.keys())[0]
    file_bytes = uploaded[file_name]

    # Decode CSV text safely across common Excel encodings
    text_data = None
    for enc in ['utf-8', 'cp1252', 'latin1']:
        try:
            text_data = file_bytes.decode(enc)
            break
        except Exception:
            pass

    # Parse multi-student CSV using standard csv.reader
    reader = csv.reader(text_data.splitlines())
    students = []
    current_student = None

    for parts in reader:
        if not parts or not any(parts):
            continue

        parts = [p.strip() for p in parts]

        # Detect student header line (e.g., "Juan Dela Cruz, BS Computer Science, 1st Year")
        if len(parts) >= 3 and parts[0].lower() not in ['code', 'subject'] and parts[2].lower() not in ['units', 'unit', '']:
            if 'code' in parts[0].lower() or 'description' in parts[1].lower():
                continue

            if not any(char.isdigit() for char in parts[0]):
                current_student = {
                    'name': parts[0],
                    'course': parts[1] if len(parts) > 1 else '',
                    'year': parts[2] if len(parts) > 2 else '',
                    'subjects': []
                }
                students.append(current_student)
                continue

        # Parse subject records under active student
        if current_student is not None:
            if parts[0].lower() in ['code', 'subject']:
                continue

            non_empty = [p for p in parts if p]
            if len(non_empty) >= 5:
                c0 = non_empty[0] # Code
                c1 = non_empty[1] # Description
                c2 = non_empty[2] # Units
                c3 = non_empty[3] # Teacher
                c4 = non_empty[4] # Grade

                try:
                    units = float(c2)
                except ValueError:
                    units = 0.0

                try:
                    grade = float(c4)
                except ValueError:
                    grade = 0.0

                current_student['subjects'].append({
                    'code': c0,
                    'desc': c1,
                    'teacher': c3,
                    'units': units,
                    'grade': grade
                })

    # Display Individual Student Reports in Console
    print("\n" + "="*80)
    print(f"Loaded File: {file_name} ({len(students)} Students Found)")
    print("="*80)

    leaderboard = []

    for idx, st in enumerate(students, 1):
        total_units = 0.0
        weighted_sum = 0.0
        subject_count = 0

        print(f"\nStudent #{idx}: {st['name']}")
        print(f"Course: {st['course']} | Year: {st['year']}")
        print("-" * 80)
        print(f"{'#':<3} {'Code':<12} {'Description':<35} {'Teacher':<20} {'Units':<6} {'Grade':<6}")
        print("-" * 80)

        for s_idx, sub in enumerate(st['subjects'], 1):
            print(f"{s_idx:<3} {sub['code']:<12} {sub['desc'][:33]:<35} {sub['teacher'][:18]:<20} {sub['units']:<6.1f} {sub['grade']:<6.2f}")
            total_units += sub['units']
            weighted_sum += (sub['units'] * sub['grade'])
            subject_count += 1

        gwa = (weighted_sum / total_units) if total_units > 0 else 0.0
        leaderboard.append({
            'name': st['name'],
            'course': st['course'],
            'year': st['year'],
            'total_units': total_units,
            'subject_count': subject_count,
            'gwa': gwa
        })

        print("-" * 80)
        print(f"Total Units: {total_units:.1f} | # of Subjects: {subject_count} | GWA: {gwa:.2f}")
        print("=" * 80)

    # Sort Leaderboard from Highest to Lowest (Lower numeric GWA is higher standing)
    leaderboard.sort(key=lambda x: x['gwa'])

    print("\n\n" + "🏆 ACADEMIC STANDING LEADERBOARD (HIGHEST TO LOWEST GWA) 🏆".center(80))
    print("=" * 80)
    print(f"{'Rank':<6} {'Student Name':<28} {'Course & Year':<28} {'Units':<7} {'GWA':<6}")
    print("-" * 80)

    for rank, st in enumerate(leaderboard, 1):
        badge = "🥇" if rank == 1 else "🥈" if rank == 2 else "🥉" if rank == 3 else "  "
        course_yr = f"{st['course']} ({st['year']})"
        print(f"{badge} {rank:<3} {st['name']:<28} {course_yr[:26]:<28} {st['total_units']:<7.1f} {st['gwa']:<6.2f}")

    print("=" * 80)

else:
    print("No file was uploaded.")

=== Upload your CSV file (e.g., Student_Grade.csv) below ===


Saving Tando_EXCEL.CSV to Tando_EXCEL.CSV

Loaded File: Tando_EXCEL.CSV (1 Students Found)

Student #1: Terence James R. Tando
Course: BS Computer Science | Year: 1st year
--------------------------------------------------------------------------------
#   Code         Description                         Teacher              Units  Grade 
--------------------------------------------------------------------------------
1   CSCC 101     INTRODUCTION TO COMPUTING           Arianne Joy Cari-A   3.0    1.50  
2   CSCC 102     FUNDAMENTALS OF PROGRAMMING         Mrs. Balala          3.0    1.00  
3   CSMATH 1     LINEAR ALGEBRA                      Victorino Jr Tobia   3.0    2.00  
4   MST 111      MATHEMATICS IN THE MODERN WORLD     Nathalie Gallaron    3.0    1.00  
5   WKL 1        INTRODUKSIYON SA FILIPINOLOHIYA (   Karen Umadhay        3.0    1.75  
6   PE 1         PATHFIT 1 - PHYSICAL ACTIVITIES T   Christian Dave Ign   2.0    1.25  
7   SOCSCI  143  RIZAL’S LIFE, WORKS AND WRITINGS 